# 📄 Gerador de Orçamento em PDF

## 🎯 Objetivo
Este notebook automatiza a geração de orçamentos profissionais em PDF a partir dos dados informados pelo usuário.

## 🔧 O que o projeto faz:
1. **Coleta** os dados do projeto (descrição, horas, valor/hora, prazo)
2. **Calcula** automaticamente o valor total do projeto
3. **Valida** as entradas para evitar erros
4. **Gera** um PDF formatado com os dados do orçamento

## 🔧 Tecnologias Utilizadas
- `fpdf` — geração de arquivos PDF com Python

---
> **Autor:** Thiago de Mattos  
> **Versão:** 2.0

## 📦 1. Instalação das Bibliotecas

Execute esta célula apenas na **primeira vez** que rodar o notebook.

In [ ]:
# Instala a biblioteca de geração de PDF
!pip install fpdf --quiet
print("✅ Bibliotecas instaladas com sucesso!")

## 📚 2. Importação das Bibliotecas

In [ ]:
# Biblioteca para geração de PDFs
from fpdf import FPDF

print("✅ Bibliotecas importadas com sucesso!")

## ⚙️ 3. Configurações Gerais

Altere as variáveis abaixo conforme necessário.

In [ ]:
# ============================================================
# CONFIGURAÇÕES — altere aqui conforme necessário
# ============================================================

# Nome do arquivo PDF gerado
NOME_ARQUIVO_PDF = "Orçamento.pdf"

# Caminho da imagem de template (deixe None se não tiver)
TEMPLATE_IMAGEM = "template.png"

# Posições do texto no PDF (ajuste conforme seu template)
POSICOES = {
    "projeto":     (115, 145),
    "horas":       (115, 160),
    "valor_hora":  (115, 175),
    "prazo":       (115, 190),
    "valor_total": (115, 205),
}

# ============================================================
print("✅ Configurações carregadas!")
print(f"   → Arquivo de saída : {NOME_ARQUIVO_PDF}")
print(f"   → Template         : {TEMPLATE_IMAGEM}")

## 📝 4. Entrada de Dados

Preencha os dados do projeto quando solicitado.

> ⚠️ **Atenção:** Horas e valor da hora devem ser números (ex: `300`, `350.50`).

In [ ]:
def coletar_dados():
    """
    Solicita e valida os dados do orçamento informados pelo usuário.

    Retorna:
        dict com os dados validados, ou None se houver erro de digitação.
    """
    try:
        # Coleta os dados do usuário
        projeto   = input("📌 Digite a descrição do projeto: ").strip()
        horas_str = input("⏱️  Digite a quantidade de horas previstas: ").strip()
        hora_str  = input("💰 Digite o valor da hora trabalhada (R$): ").strip()
        prazo     = input("📅 Digite o prazo estimado (ex: 3 meses): ").strip()

        # Validação: garante que horas e valor sejam números
        horas_previstas = float(horas_str)
        valor_hora      = float(hora_str)

        # Validação: garante que os valores sejam positivos
        if horas_previstas <= 0 or valor_hora <= 0:
            print("❌ Horas e valor da hora devem ser maiores que zero.")
            return None

        # Validação: garante que os campos de texto não estejam vazios
        if not projeto or not prazo:
            print("❌ Descrição do projeto e prazo não podem estar vazios.")
            return None

        dados = {
            "projeto":         projeto,
            "horas_previstas": horas_previstas,
            "valor_hora":      valor_hora,
            "prazo":           prazo,
        }

        print("\n✅ Dados coletados com sucesso!")
        return dados

    except ValueError:
        print("❌ Erro: Horas e valor da hora devem ser números (ex: 300, 350.50).")
        return None


# Coleta os dados do usuário
dados = coletar_dados()

## 🧮 5. Cálculo do Valor Total

In [ ]:
def calcular_valor_total(dados):
    """
    Calcula o valor total do projeto.

    Fórmula:
        Valor Total = Horas Previstas × Valor por Hora

    Parâmetros:
        dados (dict): Dicionário com os dados do orçamento.

    Retorna:
        float com o valor total calculado.
    """
    valor_total = dados['horas_previstas'] * dados['valor_hora']
    return round(valor_total, 2)


# Calcula e exibe o resumo (somente se os dados foram coletados)
if dados is not None:
    valor_total = calcular_valor_total(dados)

    print("=" * 45)
    print("  📋 RESUMO DO ORÇAMENTO")
    print("=" * 45)
    print(f"  Projeto       : {dados['projeto']}")
    print(f"  Horas Previstas: {dados['horas_previstas']}h")
    print(f"  Valor por Hora : R$ {dados['valor_hora']}")
    print(f"  Prazo Estimado : {dados['prazo']}")
    print("-" * 45)
    print(f"  💵 VALOR TOTAL  : R$ {valor_total:,.2f}")
    print("=" * 45)
else:
    print("⚠️  Execute a célula anterior para coletar os dados.")

## 📄 6. Gerar o PDF do Orçamento

> ⚠️ **Importante:** certifique-se de que o arquivo `template.png` está na mesma pasta do notebook antes de executar.

In [ ]:
import os

def gerar_pdf(dados, valor_total, nome_arquivo, template, posicoes):
    """
    Gera o arquivo PDF do orçamento.

    Parâmetros:
        dados        (dict): Dados do orçamento.
        valor_total  (float): Valor total calculado.
        nome_arquivo (str): Nome do arquivo PDF de saída.
        template     (str): Caminho da imagem de fundo (template).
        posicoes     (dict): Coordenadas (x, y) de cada campo no PDF.
    """
    try:
        # Cria o objeto PDF
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=12)

        # Adiciona a imagem de fundo (template), se existir
        if template and os.path.exists(template):
            pdf.image(template, x=0, y=0)
            print(f"🖼️  Template '{template}' aplicado.")
        else:
            print(f"⚠️  Template '{template}' não encontrado. Gerando PDF sem imagem de fundo.")

        # Preenche os campos no PDF nas posições configuradas
        pdf.text(posicoes['projeto'][0],     posicoes['projeto'][1],     dados['projeto'])
        pdf.text(posicoes['horas'][0],       posicoes['horas'][1],       str(dados['horas_previstas']))
        pdf.text(posicoes['valor_hora'][0],  posicoes['valor_hora'][1],  f"R$ {dados['valor_hora']}")
        pdf.text(posicoes['prazo'][0],       posicoes['prazo'][1],       dados['prazo'])
        pdf.text(posicoes['valor_total'][0], posicoes['valor_total'][1], f"R$ {valor_total:,.2f}")

        # Salva o arquivo PDF
        pdf.output(nome_arquivo)

        print(f"\n✅ Orçamento gerado com sucesso!")
        print(f"   → Arquivo salvo como: '{nome_arquivo}'")

    except Exception as e:
        print(f"❌ Erro ao gerar o PDF: {e}")


# Gera o PDF (somente se os dados e o valor total foram calculados)
if dados is not None and 'valor_total' in dir():
    gerar_pdf(dados, valor_total, NOME_ARQUIVO_PDF, TEMPLATE_IMAGEM, POSICOES)
else:
    print("⚠️  Execute as células anteriores primeiro.")

---

## 🚀 Pipeline Completo (Tudo de Uma Vez)

Execute a célula abaixo para rodar todo o fluxo de uma só vez:  
coleta dados → calcula total → gera PDF.

In [ ]:
def pipeline_completo():
    """
    Executa o pipeline completo de geração do orçamento em PDF.
    """
    print("=" * 45)
    print("  📄 GERADOR DE ORÇAMENTO EM PDF")
    print("=" * 45)

    # 1. Coleta os dados
    print("\n📝 Etapa 1/3 — Coletando dados...\n")
    dados = coletar_dados()
    if dados is None:
        return

    # 2. Calcula o valor total
    print("\n🧮 Etapa 2/3 — Calculando valor total...")
    valor_total = calcular_valor_total(dados)

    print("\n" + "=" * 45)
    print("  📋 RESUMO DO ORÇAMENTO")
    print("=" * 45)
    print(f"  Projeto        : {dados['projeto']}")
    print(f"  Horas Previstas: {dados['horas_previstas']}h")
    print(f"  Valor por Hora : R$ {dados['valor_hora']}")
    print(f"  Prazo Estimado : {dados['prazo']}")
    print("-" * 45)
    print(f"  💵 VALOR TOTAL  : R$ {valor_total:,.2f}")
    print("=" * 45)

    # 3. Gera o PDF
    print("\n📄 Etapa 3/3 — Gerando PDF...")
    gerar_pdf(dados, valor_total, NOME_ARQUIVO_PDF, TEMPLATE_IMAGEM, POSICOES)

    print("\n" + "=" * 45)
    print("  ✅ PIPELINE CONCLUÍDO COM SUCESSO!")
    print("=" * 45)


# Executa o pipeline completo
pipeline_completo()